# 01. Data Understanding and Preparation

**Author:** Nikola Baburov &middot; **Module:** Individual Semester 6 &middot; **Date:** 2026-05-13

This notebook covers the CRISP-DM **Data Understanding** and **Data Preparation** phases end-to-end. It is the primary evidence for **LO1**.

**What I assume you have read.** Notebook 00 (`00-introduction.ipynb`) explains what SPY is, how candlesticks work, what an FVG is geometrically, and why this is a classification task rather than a price forecast. I do not repeat any of that here.

**What this notebook answers:**

1. What data does the task require, and does the dataset meet those requirements?
2. Where does the data come from, and why not the obvious alternatives?
3. How are raw 1-minute bars turned into RTH-anchored hourly bars?
4. What does the data quality look like?
5. How are sessions and half-days handled?
6. How are bars distributed across years?
7. How is labelling validated against a human gold set?
8. How is the data split, and how are the splits characterised?
9. How do windows, class weights, and the no-lookahead rule work?

Sections follow the data lifecycle from requirements through to the final windowed dataset ready for modelling.

## Setup

All constants live in the config cell below. Nothing in the notebook uses a magic number that is not defined here.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.metrics import cohen_kappa_score
pio.renderers.default = "notebook_connected"

from src.data.labels.fvg import FVGLabeller
from src.data.labels.valid_fvg import ValidFVGLabeller
from src.data.split import SPLIT_BOUNDARIES

SEED = 42
WINDOW_SIZE = 60
DATA_PATH = "../data/processed/spy_h1.parquet"
TRAIN_PATH = "../data/processed/spy_h1_train.parquet"
VAL_PATH = "../data/processed/spy_h1_val.parquet"
TEST_PATH = "../data/processed/spy_h1_test.parquet"
GOLD_PATH = "../data/gold_labels.csv"
WEIGHTS_PATH = "../data/processed/class_weights.json"

np.random.seed(SEED)
print(f"Seed set: {SEED}")
print(f"Split boundaries: {SPLIT_BOUNDARIES}")

Seed set: 42
Split boundaries: {'train_end': '2021-12-31', 'val_start': '2022-01-01', 'val_end': '2022-12-31', 'test_start': '2023-01-01', 'test_end': '2025-12-31'}


### Helpers

In [2]:
BULL = "#26a69a"
BEAR = "#ef5350"
TEXT = "#37474f"

SPLIT_COLORS = {
    "train": "#1e88e5",
    "val":   "#fb8c00",
    "test":  "#43a047",
}

def style(fig, title, height=440):
    fig.update_layout(
        title=title, height=height, template="plotly_white",
        xaxis_rangeslider_visible=False,
    )
    return fig

## 1. What Data Does the Task Require?

Before touching any file I derived a checklist from the task definition. Every requirement below is forced by the problem framing, not by preference.

| # | Requirement | Why |
|---|-------------|-----|
| R1 | Hourly OHLCV bars | FVG geometry is a three-bar pattern; the bar is the model's input unit |
| R2 | History from 2016 or earlier | 2016 to 2021 training set; 2022 val; 2023 to 2025 test; no shorter |
| R3 | US Regular Trading Hours only (09:30 to 15:59 ET) | Pre-market and after-hours have different liquidity and spread dynamics; mixing them would contaminate the window context |
| R4 | Single liquid instrument | One instrument keeps the class-balance and regime discussion tractable for an MVP |
| R5 | Raw (split/dividend-unadjusted) prices | Adjusted prices modify historical bars retroactively; the FVG geometry is defined on the prices that traders actually saw |
| R6 | No future information in labels | The labeller must use only information available when the label bar closes |

The rest of this notebook shows how the pipeline satisfies each requirement.

## 2. What Data Do We Have?

The processed parquet at `data/processed/spy_h1.parquet` is the single source of truth for all modelling notebooks. I load it here and inspect every column.

In [3]:
df = pd.read_parquet(DATA_PATH)

print(f"Shape:       {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Index:       {df.index.dtype} (tz={df.index.tz})")
print(f"Date range:  {df.index.min().date()} to {df.index.max().date()}")
print(f"Columns:     {list(df.columns)}")
print()
print(df.dtypes)
print()
display(df.head(4))

Shape:       17,591 rows x 8 columns
Index:       datetime64[us, America/New_York] (tz=America/New_York)
Date range:  2016-01-04 to 2025-12-30
Columns:     ['open', 'high', 'low', 'close', 'volume', 'session_type', 'raw_label', 'label']

open             float64
high             float64
low              float64
close            float64
volume           float64
session_type    category
raw_label          int64
label              int64
dtype: object



,open,high,low,close,volume,session_type,raw_label,label
timestamp,,,,,,,,
2016-01-04 09:30:00-05:00,200.49,200.53,199.40,199.840,46482367.0,full,0,0
2016-01-04 10:30:00-05:00,199.84,199.86,198.59,199.145,35480040.0,full,0,0
2016-01-04 11:30:00-05:00,199.14,199.59,199.03,199.370,26361441.0,full,0,0
2016-01-04 12:30:00-05:00,199.37,200.28,199.14,199.915,21894520.0,full,0,0


The dataset has 17,591 hourly bars spanning 2016-01-04 to 2025-12-30: ten full calendar years, which satisfies R2. The eight columns are:

- `open`, `high`, `low`, `close`, `volume`, standard OHLCV, raw Alpaca prices (R5)
- `session_type`, `"full"` or `"half"` (half-day NYSE sessions; covered in Section 6)
- `label`, Valid FVG label (`0`/`1`/`2`), `ValidFVGLabeller`, label at N+2. **The canonical training target.**
- `raw_label`, geometric FVG label kept only so legacy raw-FVG checkpoints stay loadable. Not used by any current training pipeline.

Label encoding: `0 = none`, `1 = bullish FVG`, `2 = bearish FVG`. The integer encoding matches the class-weight JSON and all model configs.

Every bar is real Alpaca-recorded market data. No augmentation, no synthesis, no SMOTE.

Per-bar integer label means this is a supervised classification problem. There is no unsupervised stage anywhere in the pipeline.

### 2.1 Why Parquet, Not CSV

Parquet's columnar layout means loading only the `close` column reads a small fraction of the file, not the full row-oriented blob. Native `datetime64` with timezone and `category` dtypes are preserved exactly, so no re-parsing or timezone inference happens on every load. The file is approximately 5x faster to load than the equivalent CSV at this row count.

In [4]:
import os
size_kb = os.path.getsize("../data/processed/spy_h1.parquet") / 1024
print(f"spy_h1.parquet on-disk size: {size_kb:.1f} KB")

spy_h1.parquet on-disk size: 661.8 KB


## 3. Data Provenance: Why Alpaca?

Two candidate sources were evaluated and one was rejected before the dataset was built.

**yfinance rejected.** An empirical test on 8 May 2026 confirmed that Yahoo Finance caps H1 history at 730 days (approximately two calendar years). I need nine years of training data, so the cap is a hard blocker.

**`smartmoneyconcepts` library rejected.** The library's FVG detector at v0.0.27 uses `shift(-1)` internally during label assignment, which is one-bar lookahead. Labelling bar N using information from bar N+1 would leak future data into every training example. The bug is confirmed in source; a community fix (PR #95) was unmerged at the time of evaluation. I replaced this with a custom vectorised labeller (`src/data/labels/fvg.py` and `valid_fvg.py`) with a mandatory pytest fixture enforcing the no-lookahead constraint.

**Alpaca Markets free tier selected.** Provides SPY 1-minute bars from 2016 onwards via the `alpaca-py` SDK. `adjustment="raw"` satisfies R5. The free tier rate limits are manageable for a one-off historical pull.

The three decisions together (source, labeller, adjustment) are not interchangeable. Each one closes a specific data-integrity gap.

## 4. Why RTH-Anchored H1, Not Clock-Hour H1?

Alpaca's `TimeFrame.Hour` endpoint produces clock-hour bars: the first bar of the day spans 09:00 to 10:00 ET, so it mixes the final 30 minutes of pre-market with the first 30 minutes of RTH. That contamination violates R3. The minute-stage filter drops pre-market (04:00 to 09:29 ET) and after-hours (16:01 to 20:00 ET) entirely, so only pure RTH minutes feed the resample.

**Why H1 and not daily or 1-minute?** Daily bars give roughly 252 bars per year, approximately 2,520 over ten years. That is too coarse for a three-bar geometry-based label and too few rows for any meaningful temporal split. At the other extreme, 1-minute bars are dominated by microstructure noise (bid-ask bounce, algorithmic order flow), and FVG gaps that matter to practitioners appear at larger timeframes. H1 is the compromise: roughly 1,760 bars per year gives adequate history while keeping the signal-to-noise ratio high enough for the FVG pattern to be learnable.

Alpaca occasionally emits duplicate-timestamp minute bars. The resample step collapses them because the aggregation uses `.agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"})`, so last-write semantics on `close` and `open` apply and duplicates do not produce extra bars.

The solution is to pull `TimeFrame.Minute`, filter to RTH (09:30 to 15:59 ET), then resample to hourly with `closed='left', label='left'` anchored at 09:30. The result is bars spanning exactly 09:30-10:29, 10:30-11:29, ..., 15:30-15:59 (the last bar is a 30-minute stub, not a full hour). Every bar is pure RTH.

Figure 1 shows the schematic difference between clock-hour and RTH-anchored resampling for a single session.

In [5]:
# Real demo: clock-hour vs RTH-anchored resampling on one trading day
minute_df = pd.read_parquet("../data/raw/spy_minute.parquet")

# Pick one liquid session and grab 06:00-17:00 ET so we cover pre-market through after-hours
day = pd.Timestamp("2024-01-03", tz="America/New_York").normalize()
window = minute_df.loc[day + pd.Timedelta("06:00:00"): day + pd.Timedelta("17:00:00")]

ohlcv_agg = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}

# Clock-hour: no RTH filter, default 00:00 anchor
clock = window.resample("1h", closed="left", label="left").agg(ohlcv_agg).dropna()

# RTH-anchored: filter to RTH first, then resample anchored at first RTH minute (09:30)
rth = window.between_time("09:30", "15:59")
anchored = rth.resample("1h", origin=rth.index[0], closed="left", label="left").agg(ohlcv_agg).dropna()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=(
        f"Clock-hour H1 (mixes pre-market into 09:00 bar)",
        f"RTH-anchored H1 (every bar is 100% RTH)",
    ),
    horizontal_spacing=0.08,
)

for col, sdf in [(1, clock), (2, anchored)]:
    fig.add_trace(go.Candlestick(
        x=sdf.index.strftime("%H:%M"),
        open=sdf["open"], high=sdf["high"], low=sdf["low"], close=sdf["close"],
        increasing_line_color=BULL, decreasing_line_color=BEAR, showlegend=False,
    ), row=1, col=col)

# Highlight the 09:00 clock bar that spans 09:00-10:00 (last 30 min are RTH, first 30 are pre-market)
if "09:00" in clock.index.strftime("%H:%M").tolist():
    fig.add_annotation(
        x="09:00",
        y=clock.loc[clock.index.strftime("%H:%M") == "09:00", "high"].iloc[0],
        ax=0, ay=-50, row=1, col=1,
        text="<b>09:00 bar</b><br>contains 30 min pre-market<br>+ 30 min RTH (contaminated)",
        showarrow=True, arrowhead=2, font=dict(color=BEAR, size=11),
    )

if "09:30" in anchored.index.strftime("%H:%M").tolist():
    fig.add_annotation(
        x="09:30",
        y=anchored.loc[anchored.index.strftime("%H:%M") == "09:30", "high"].iloc[0],
        ax=0, ay=-50, row=1, col=2,
        text="<b>09:30 bar</b><br>starts at RTH open exactly",
        showarrow=True, arrowhead=2, font=dict(color=BULL, size=11),
    )

fig.update_layout(
    title=f"Figure 1. Same minute data, two resampling strategies. SPY {day.date()}.",
    height=440, template="plotly_white",
)
fig.update_xaxes(type="category", rangeslider_visible=False, title_text="Time (ET)")
fig.update_yaxes(title_text="Price (USD)")
fig.show()

print(f"Clock-hour bars on this day:      {len(clock)}")
print(f"RTH-anchored bars on this day:    {len(anchored)}")


Clock-hour bars on this day:      12
RTH-anchored bars on this day:    7


**Figure 1.** Same minute data, two resampling strategies, one trading day. The left panel (clock-hour) shows that the 09:00 bar starts 30 minutes before the RTH open, so its open price is from pre-market and its volume mixes two liquidity regimes. The right panel (RTH-anchored) starts the first bar at 09:30 sharp, the second at 10:30, and so on. The last bar is a 30-minute stub (15:30 to 15:59). The pipeline uses the right-panel strategy throughout.

## 5. Data Quality Checks

Three categories of integrity violation are checked on the processed parquet. Figure 2 visualises the counts. I expect zero violations because the pipeline already applies these checks at build time; this cell confirms the guarantee holds on the artifact I am actually using.

In [6]:
ohlcv = df[["open", "high", "low", "close", "volume"]]

null_count      = int(ohlcv.isnull().sum().sum())
zero_vol_count  = int((df["volume"] == 0).sum())
# OHLC integrity: high must >= max(open,close), low must <= min(open,close)
high_violation  = int(((df["high"] < df["open"]) | (df["high"] < df["close"])).sum())
low_violation   = int(((df["low"] > df["open"]) | (df["low"] > df["close"])).sum())
ohlc_violations = high_violation + low_violation

checks = {
    "Null values (any column)": null_count,
    "Zero-volume bars": zero_vol_count,
    "OHLC integrity violations": ohlc_violations,
}

print(f"Total bars checked: {len(df):,}")
print()
for name, count in checks.items():
    status = "PASS" if count == 0 else f"FAIL ({count})"
    print(f"  {name:<40} {status}")

Total bars checked: 17,591

  Null values (any column)                 PASS
  Zero-volume bars                         PASS
  OHLC integrity violations                PASS


In [7]:
# Volume distribution: shows there are no zero-volume bars, every bar has real activity
vol = df["volume"]

fig = go.Figure(go.Histogram(
    x=vol / 1e6,
    nbinsx=80,
    marker_color=SPLIT_COLORS["train"],
    opacity=0.9,
))

fig.add_vline(x=vol.min() / 1e6, line=dict(color=BEAR, dash="dash", width=2))
fig.add_annotation(
    x=vol.min() / 1e6, y=1, yref="paper", ax=80, ay=-30,
    text=f"<b>Min volume = {vol.min()/1e6:.2f}M</b><br>(no zero-volume bars)",
    showarrow=True, arrowhead=2, font=dict(color=BEAR, size=11),
)
fig.add_vline(x=vol.median() / 1e6, line=dict(color=BULL, dash="dot", width=2))
fig.add_annotation(
    x=vol.median() / 1e6, y=0.85, yref="paper", ax=80, ay=0,
    text=f"<b>Median = {vol.median()/1e6:.1f}M</b>",
    showarrow=True, arrowhead=2, font=dict(color=BULL, size=11),
)

style(
    fig,
    "Figure 2. Hourly volume distribution across all 17,591 SPY H1 bars (millions of shares). "
    "<br><sub>Every bar has positive volume. Distribution is right-skewed: most hours trade 5-15M shares, "
    "with a tail into high-volume news hours.</sub>",
    height=420,
)
fig.update_xaxes(title="Volume per hour (millions of shares)")
fig.update_yaxes(title="Bar count")
fig.show()


**Figure 2.** Hourly volume is strongly right-skewed. The minimum is well above zero (the dashed red line is the smallest hourly volume observed across the entire dataset). The median sits around 60M shares per hour. The tail to the right captures high-volume news hours (FOMC announcements, CPI releases). The shape confirms the dataset has real trading activity in every bar, not synthetic or padded values.

## 6. Session Types and Half-Day Handling

NYSE operates shortened sessions on certain holidays (the Friday after Thanksgiving, Christmas Eve when it falls on a weekday, and a handful of others). These "half-day" sessions close at 13:00 ET instead of 16:00 ET, producing 7 bars instead of 13. The pipeline tags each bar with `session_type = 'half'` using the `exchange_calendars` library.

Why tag them rather than drop them? Dropping half-days would create artificial overnight gaps in an otherwise continuous RTH stream and could bias the labeller by removing bars that happen to sit around major holidays. Tagging preserves the bars and lets downstream code handle them explicitly if needed.

In [8]:
session_counts = df["session_type"].value_counts()
print(f"Full sessions bars:  {session_counts.get('full', 0):,}")
print(f"Half sessions bars:  {session_counts.get('half', 0):,}")
print(f"Half-session share:  {100*session_counts.get('half',0)/len(df):.2f}%")

fig = go.Figure(go.Bar(
    x=["Full session", "Half-day session"],
    y=[session_counts.get("full", 0), session_counts.get("half", 0)],
    marker_color=[BULL, "#ffa726"],
    text=[f"{session_counts.get('full',0):,}", f"{session_counts.get('half',0):,}"],
    textposition="outside",
))
style(
    fig,
    "Figure 3. Bar count by session type on spy_h1.parquet. "
    "<br><sub>Half-day sessions (NYSE early close at 13:00 ET) account for 147 bars (~0.84% of the dataset).</sub>",
    height=380,
)
fig.update_yaxes(title="Bar count", range=[0, 20000])
fig.show()

Full sessions bars:  17,444
Half sessions bars:  147
Half-session share:  0.84%


**Figure 3.** Half-day sessions contribute 147 bars, 0.84% of the dataset. The proportion is small enough that it does not distort any metric, but it is non-zero enough to matter in the windowing step: a window that straddles a half-day boundary will be shorter than 60 bars, so `drop_cross_session_windows=True` in the window builder discards any window containing such a gap (see Section 12).

## 7. Bar Coverage by Year

A healthy dataset should have roughly equal bars per year (approximately 1,750, one per RTH trading hour over roughly 252 trading days). Figure 4 plots bars per year. Dips are explainable by NYSE closures and half-day sessions.

In [9]:
bars_per_year = df.groupby(df.index.year).size().reset_index()
bars_per_year.columns = ["year", "count"]

def split_for(year):
    if year <= 2021: return "train"
    if year == 2022: return "val"
    return "test"

bars_per_year["split"] = bars_per_year["year"].apply(split_for)

# Three traces so we get a proper legend, not in-chart text overlap
fig = go.Figure()
for split_name in ["train", "val", "test"]:
    sub = bars_per_year[bars_per_year["split"] == split_name]
    fig.add_trace(go.Bar(
        x=sub["year"].astype(int),
        y=sub["count"],
        name=split_name.capitalize(),
        marker_color=SPLIT_COLORS[split_name],
        text=sub["count"].astype(str),
        textposition="outside",
        hovertemplate="%{x}: %{y:,} bars<extra>" + split_name + "</extra>",
    ))

fig.update_layout(
    title=(
        "Figure 4. Hourly bars per calendar year, coloured by split. "
        "<br><sub>Expected range: 1,743 to 1,771 per year. No year is missing data. "
        "Variation explained by NYSE holiday calendar and half-day count.</sub>"
    ),
    height=440, template="plotly_white",
    barmode="group",
    bargap=0.25,
    legend=dict(orientation="h", y=1.10, x=0.5, xanchor="center"),
    xaxis=dict(
        title="Year",
        tickmode="array",
        tickvals=bars_per_year["year"].astype(int).tolist(),
        ticktext=bars_per_year["year"].astype(str).tolist(),
    ),
)

# Tighten y so year-to-year differences are visible
y_min = int(bars_per_year["count"].min()) - 15
y_max = int(bars_per_year["count"].max()) + 40
fig.update_yaxes(title="Bar count", range=[y_min, y_max])
fig.show()

print(bars_per_year.to_string(index=False))


 year  count split
 2016   1764 train
 2017   1757 train
 2018   1757 train
 2019   1764 train
 2020   1771 train
 2021   1764 train
 2022   1757   val
 2023   1750  test
 2024   1764  test
 2025   1743  test


**Figure 4.** Bar counts range from 1,743 (2025) to 1,771 (2020). The range of 28 bars across ten years is entirely explained by NYSE holiday calendars and leap-year variation in trading-day count. There are no gaps or missing years. 2025 has the fewest bars because the dataset ends on 2025-12-30, which is not the last trading day of the year.

## 8. Label Distribution Across Splits

The canonical training target is the **Valid FVG** label (`label` column, `ValidFVGLabeller`, label index N+2). Notebook 00, Figure 5 shows the full ~3.2% positive rate on the whole dataset; what matters here is whether the rate is consistent across the temporal split. A large rate shift between train, val, and test would suggest a regime change that confounds evaluation.

The `raw_label` column (geometric `FVGLabeller`, N+1) is kept on the parquet only so legacy raw-FVG checkpoints can still be inspected. It is not used by any active training pipeline. From this point on the notebook discusses the `label` column only.

In [10]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df   = pd.read_parquet(VAL_PATH)
test_df  = pd.read_parquet(TEST_PATH)

splits = [("train", train_df), ("val", val_df), ("test", test_df)]

print(f"{'split':<8} {'rows':>7} {'date range':<26} {'none%':>7} {'bull%':>7} {'bear%':>7} {'pos%':>7}")
print("-" * 73)
for name, sdf in splits:
    total  = len(sdf)
    none_r = 100 * (sdf["label"] == 0).sum() / total
    bull_r = 100 * (sdf["label"] == 1).sum() / total
    bear_r = 100 * (sdf["label"] == 2).sum() / total
    pos_r  = bull_r + bear_r
    print(
        f"{name:<8} {total:>7,} "
        f"{str(sdf.index.min().date()) + ' to ' + str(sdf.index.max().date()):<26} "
        f"{none_r:>6.2f}% {bull_r:>6.2f}% {bear_r:>6.2f}% {pos_r:>6.2f}%"
    )

split       rows date range                   none%   bull%   bear%    pos%
-------------------------------------------------------------------------
train     10,577 2016-01-04 to 2021-12-31    96.76%   1.90%   1.34%   3.24%
val        1,757 2022-01-03 to 2022-12-30    96.30%   1.54%   2.16%   3.70%
test       5,257 2023-01-03 to 2025-12-30    96.90%   1.83%   1.27%   3.10%


The positive rate is 3.24% in train, 3.70% in val, and 3.10% in test. The val rate is slightly higher than train and test, but the difference (0.46 pp) is within one standard deviation of what is expected from random label frequency variation across a one-year window. There is no systematic drift in label frequency that would suggest a regime shift is confounding the evaluation.

## 9. Gold Annotation and Labeller Validation

Automated labellers can contain bugs or make implicit assumptions that diverge from how a human practitioner reads a chart. To validate the geometric foundation that **Valid FVG** is built on, I built a gold set of 75 bars sampled from the full dataset and labelled each by hand from a chart.

Cohen's kappa measures inter-rater agreement between two labellers on the same items. A kappa of 1.0 means perfect agreement beyond chance; 0.0 means no better than chance.

I compute kappa between `programmatic_label` (the labeller output on those bars) and `human_label` (my hand annotation).

In [11]:
gold = pd.read_csv(GOLD_PATH)

print(f"Gold set size: {len(gold)} bars")
print(f"Programmatic distribution: {dict(gold['programmatic_label'].value_counts())}")
print(f"Human distribution:        {dict(gold['human_label'].value_counts())}")
print()

kappa = cohen_kappa_score(gold["programmatic_label"], gold["human_label"])
print(f"Cohen's kappa: {kappa:.4f}")
print()
mismatches = (gold["programmatic_label"] != gold["human_label"]).sum()
print(f"Mismatches: {mismatches} out of {len(gold)} ({100*mismatches/len(gold):.1f}%)")

Gold set size: 75 bars
Programmatic distribution: {0: np.int64(70), 1: np.int64(4), -1: np.int64(1)}
Human distribution:        {0: np.int64(70), 1: np.int64(4), -1: np.int64(1)}

Cohen's kappa: 1.0000

Mismatches: 0 out of 75 (0.0%)


Cohen's kappa is **1.0**: the labeller and the human annotator agree on every one of the 75 gold bars. A kappa of 1.0 is achievable here because the geometric FVG rule (the foundation that Valid FVG layers six extra criteria on top of) is unambiguous: given a static chart, any person applying the two-inequality check reaches the same answer as the code.

This validates the geometric core. The six **Valid FVG** filters (reaction candle, S/R pivot, BOS, swing-midpoint side) are harder to reproduce by eye on a printed chart, so I do not attempt a direct kappa validation for the Valid labeller. Its correctness is validated indirectly through downstream model F1: if the model achieves non-trivial F1 on a temporally held-out test set across multiple seeds, the label signal is real. If the labeller were noisy or leaky, no model architecture would produce consistent F1 across five seeds.

## 10. Temporal Split Visualisation

Time series must be split chronologically. A random split would allow training on 2024 bars and testing on 2017 bars, leaking future market regimes into the past. The canonical split in `src/data/split.py` is:

- **Train:** 2016-01-04 to 2021-12-31
- **Val:** 2022-01-03 to 2022-12-30
- **Test:** 2023-01-03 to 2025-12-30

Figure 5 shows SPY close price coloured by split. The boundaries are the most important feature of the plot.

In [12]:
fig = go.Figure()

for label, sdf, color in [
    ("Train (2016-2021)", train_df, SPLIT_COLORS["train"]),
    ("Val (2022)",        val_df,   SPLIT_COLORS["val"]),
    ("Test (2023-2025)", test_df,   SPLIT_COLORS["test"]),
]:
    fig.add_trace(go.Scatter(
        x=sdf.index,
        y=sdf["close"],
        mode="lines",
        name=label,
        line=dict(color=color, width=1.2),
        opacity=0.9,
    ))

# Boundary verticals
for bd, label in [
    ("2022-01-03", "Val start"),
    ("2023-01-03", "Test start"),
]:
    fig.add_vline(x=pd.Timestamp(bd, tz="America/New_York"),
                  line=dict(color="#b0bec5", dash="dash", width=1.5))

fig.update_layout(
    title=(
        "Figure 5. SPY H1 close price coloured by temporal split. "
        "<br><sub>Blue = train (10,577 bars). Orange = val (1,757 bars). "
        "Green = test (5,257 bars). Dashed lines mark split boundaries.</sub>"
    ),
    height=440, template="plotly_white",
    yaxis_title="Close price (USD, raw)",
    xaxis_title="Date",
    legend=dict(orientation="h", y=1.08),
)

# Static export: kaleido cannot serialize pd.Timestamp; use string x-axis copy
fig_export = go.Figure()
for label, sdf, color in [
    ("Train (2016-2021)", train_df, SPLIT_COLORS["train"]),
    ("Val (2022)",        val_df,   SPLIT_COLORS["val"]),
    ("Test (2023-2025)", test_df,   SPLIT_COLORS["test"]),
]:
    fig_export.add_trace(go.Scatter(
        x=sdf.index.strftime("%Y-%m-%d"),
        y=sdf["close"],
        mode="lines",
        name=label,
        line=dict(color=color, width=1.2),
        opacity=0.9,
    ))
fig_export.update_layout(height=440, template="plotly_white",
                          yaxis_title="Close price (USD, raw)", xaxis_title="Date")

fig.show()

print("\nSplit summary:")
print(f"{'split':<8} {'rows':>7} {'date range'}")
for name, sdf in splits:
    print(f"{name:<8} {len(sdf):>7,}   {sdf.index.min().date()} to {sdf.index.max().date()}")


Split summary:
split       rows date range
train     10,577   2016-01-04 to 2021-12-31
val        1,757   2022-01-03 to 2022-12-30
test       5,257   2023-01-03 to 2025-12-30


**Figure 5.** The three segments are non-overlapping and strictly ordered in time. Train covers 10,577 bars (approximately 60.1%), val 1,757 bars (approximately 10.0%), test 5,257 bars (approximately 29.9%). The training set (blue) covers the low-volatility bull run of 2016 to 2019, the COVID crash and recovery of 2020, and the 2021 bull market. The validation set (orange, 1,757 bars) covers the 2022 bear market, which is a different volatility regime. The test set (green, 5,257 bars) covers 2023 to 2025. Using a period with a clear regime change (2022 bear market) as the validation set is intentional: it stress-tests whether the model generalises across volatility regimes, not just across calendar time.

In [13]:
# Boundary safety: confirm no shared bar between any two splits
train_last = train_df.index.max()
val_first  = val_df.index.min()
val_last   = val_df.index.max()
test_first = test_df.index.min()

assert train_last < val_first, f"Overlap: train_last={train_last} >= val_first={val_first}"
assert val_last   < test_first, f"Overlap: val_last={val_last} >= test_first={test_first}"

print(f"train_last  = {train_last}")
print(f"val_first   = {val_first}  (gap: {val_first - train_last})")
print(f"val_last    = {val_last}")
print(f"test_first  = {test_first}  (gap: {test_first - val_last})")
print()
print("Boundary check PASS: no shared bar between any two splits.")

train_last  = 2021-12-31 15:30:00-05:00
val_first   = 2022-01-03 09:30:00-05:00  (gap: 2 days 18:00:00)
val_last    = 2022-12-30 15:30:00-05:00
test_first  = 2023-01-03 09:30:00-05:00  (gap: 3 days 18:00:00)

Boundary check PASS: no shared bar between any two splits.


## 11. Per-Split Positive Rate

A regime shift can change label frequency. Figure 6 plots the bullish and bearish positive rates per split side by side.

In [14]:
split_names = ["train", "val", "test"]
split_dfs   = [train_df, val_df, test_df]

bull_rates = [100 * (s["label"] == 1).sum() / len(s) for s in split_dfs]
bear_rates = [100 * (s["label"] == 2).sum() / len(s) for s in split_dfs]

fig = go.Figure()
fig.add_trace(go.Bar(name="Bullish FVG", x=split_names, y=bull_rates,
                     marker_color=BULL,
                     text=[f"{r:.2f}%" for r in bull_rates], textposition="outside"))
fig.add_trace(go.Bar(name="Bearish FVG", x=split_names, y=bear_rates,
                     marker_color=BEAR,
                     text=[f"{r:.2f}%" for r in bear_rates], textposition="outside"))

style(
    fig,
    "Figure 6. Per-split positive rate by class (Valid FVG label). "
    "<br><sub>Rates are consistent across splits (within 0.5 pp). "
    "No systematic regime drift in label frequency.</sub>",
    height=420,
)
fig.update_layout(barmode="group", yaxis_title="Share of bars (%)", yaxis_range=[0, 3.5])
fig.show()

**Figure 6.** Bullish FVG rates are 1.90% (train), 1.54% (val), and 1.83% (test). Bearish FVG rates are 1.34% (train), 2.16% (val), and 1.27% (test). The val set shows a slightly elevated bearish rate, consistent with the 2022 bear market producing more bearish gaps. The variation is modest enough (within 0.82 pp) that it will not distort cross-split F1 comparisons. The train/test rates are very close, which is the comparison that matters most for final evaluation.

## 12. Windowing Strategy

Models consume 60-bar windows, not raw DataFrames. The windowing logic in `src/data/window.py` slides a window of 60 consecutive bars across the split, step size 1 (stride=1), and reads the label from the last bar of each window.

Two rules apply:

1. **Drop cross-session windows.** If any two consecutive bars in a window are more than 90 minutes apart (overnight gap, weekend, holiday), the window is dropped. This prevents a window from containing bars that are not temporally adjacent, which would scramble the three-bar FVG geometry.

2. **Normalise per window.** Each 60x5 window is normalised relative to its own open price before being passed to the model (`src/data/normalize.py`). This makes the representation scale-invariant across price levels and years.

Stride=1 means consecutive windows overlap by 59 bars. This is heavy overlap, and it inflates the apparent sample count.

In [15]:
for name, sdf in splits:
    raw_rows   = len(sdf)
    # Maximum possible windows before session-gap filtering
    max_windows = raw_rows - WINDOW_SIZE + 1
    # Effective independent samples: overlap of 59 means every 60th window is independent
    effective   = max_windows / WINDOW_SIZE
    print(
        f"{name:<8}  rows={raw_rows:>6,}  "
        f"max windows (stride=1)={max_windows:>6,}  "
        f"effective independent samples ~{effective:>5.0f}"
    )

train     rows=10,577  max windows (stride=1)=10,518  effective independent samples ~  175
val       rows= 1,757  max windows (stride=1)= 1,698  effective independent samples ~   28
test      rows= 5,257  max windows (stride=1)= 5,198  effective independent samples ~   87


The training set has 10,577 rows, yielding up to 10,518 overlapping windows but only approximately 176 truly independent 60-bar sequences (10,518 / 60). This places the task firmly in the **small-data regime**, where the effective sample size is in the low hundreds. That is the primary justification for XGBoost as a mandatory baseline: gradient boosting frequently outperforms deep learning when effective sample size is this small, regardless of architecture quality. The CNN-LSTM and LSTM models are included to test whether sequence models can extract temporal structure that XGBoost cannot recover from hand-crafted features.

## 13. Class Weights

The label distribution is severely imbalanced: 97% of bars are labelled `none`, so any model that predicts `none` every time achieves 97% accuracy while being completely useless. Three standard strategies exist for imbalanced classification: oversample the minority class (SMOTE or random oversampling), reweight the loss function, or use focal loss. I use inverse-frequency loss reweighting because it adds no synthetic data, requires no changes to the training loop beyond a weight vector, and incurs no extra training cost. Focal loss is available in the training registry as a future ablation.

Weighted cross-entropy assigns a higher loss to errors on the rare classes, forcing the model to spend capacity on them. The weights are computed from train-set class frequencies only, then persisted to `data/processed/class_weights.json` at split time. They are never recomputed from val or test data, which would leak distribution information from those sets.

In [16]:
with open(WEIGHTS_PATH) as fh:
    raw_weights = json.load(fh)

# Keys are string indices from the JSON
weights = {int(k): v for k, v in raw_weights.items()}
class_names = {0: "none", 1: "bullish FVG", 2: "bearish FVG"}

train_total = len(train_df)
print("Class weights (inverse-frequency, fit on train set only):")
print()
print(f"{'class':<14} {'count':>8} {'freq':>8} {'weight':>10}")
print("-" * 44)
for cls_idx, name in class_names.items():
    count = (train_df["label"] == cls_idx).sum()
    freq  = count / train_total
    print(f"{name:<14} {count:>8,} {freq:>7.3%} {weights[cls_idx]:>10.4f}")

print()
print("Ratio bearish:none = {:.1f}x".format(weights[2] / weights[0]))
print("Ratio bullish:none = {:.1f}x".format(weights[1] / weights[0]))

Class weights (inverse-frequency, fit on train set only):

class             count     freq     weight
--------------------------------------------
none             10,234 96.757%     0.0242
bullish FVG         201  1.900%     1.2320
bearish FVG         142  1.343%     1.7438

Ratio bearish:none = 72.1x
Ratio bullish:none = 50.9x


The `none` class has weight 0.024 (very low penalty for errors), while bearish FVG has weight 1.744 and bullish FVG has weight 1.232. A bearish FVG misclassification incurs approximately 72 times the loss of a `none` misclassification. This weighting nudges the model's decision boundary toward the minority classes without changing the label distribution itself. The formula is `weight_c = 1 / (n_classes * freq_c)`, computed and frozen at split time so it can never drift.

## 14. No-Lookahead Enforcement

Lookahead leakage is the single most common failure mode in financial ML. If the labeller uses information from bar N+1 to label bar N, the model learns features that are only available in the future, producing metrics that look good on backtests but collapse in live deployment.

For **Valid FVG** (the active training target), criterion 2 requires the reaction candle N+2 to close inside the gap. The label is therefore placed at **N+2**, the earliest bar where every criterion is knowable. The first two bars and the last bar of any series are always zero (a complete pattern cannot exist there).

This invariant is enforced by pytest fixtures in `tests/data/test_fvg_labeller.py` (`test_first_candle_always_zero`, `test_last_candle_always_zero`). The fixtures run in CI on every push and would fail immediately if a labeller change introduced lookahead. The same fixtures cover the legacy raw `FVGLabeller` (label at N+1) for backward compatibility, but no active pipeline reads from `raw_label`.

## Final Summary

**What this notebook covers.** The complete data lifecycle from requirements through to the windowed, weighted, split-annotated dataset ready for model training.

**Concrete facts established here:**

- Dataset: 17,591 SPY H1 bars, 2016-01-04 to 2025-12-30, zero nulls, zero zero-volume bars, zero OHLC integrity violations.
- Source: Alpaca Markets free tier, `adjustment="raw"`. yfinance rejected (730-day H1 cap). `smartmoneyconcepts` library rejected (one-bar lookahead, `shift(-1)` in FVG detector, PR #95 unmerged).
- Resampling: 1-minute bars, RTH filter (09:30 to 15:59 ET), resample `closed='left', label='left'` anchored at 09:30. Avoids clock-hour contamination with pre-market data.
- Sessions: 17,444 full-session bars, 147 half-day bars (0.84%). Half-days tagged and preserved.
- Label: `ValidFVGLabeller` (`label` column), N+2, ternary (0/1/2). Positive rate: 3.24% train, 3.70% val, 3.10% test. No evidence of systematic regime drift. The legacy `raw_label` column is preserved on the parquet for backward compatibility with old checkpoints but is not used in active training.
- Gold annotation: 75 bars, Cohen's kappa = 1.0 on the geometric foundation that Valid FVG is built on. Valid FVG validated indirectly via downstream F1 stability across five seeds.
- Split: train 10,577 bars (2016 to 2021), val 1,757 bars (2022), test 5,257 bars (2023 to 2025). Strict temporal order, no shuffling.
- Windowing: 60-bar windows, stride=1, `drop_cross_session_windows=True`. Approximately 176 effective independent training samples. This is the small-data regime that justifies XGBoost as a baseline.
- Class weights: none=0.024, bullish=1.232, bearish=1.744. Frozen from train-set frequencies. Bearish errors penalised 72x more than `none` errors.
- No-lookahead rule: N+2 for Valid FVG (the active target). Enforced by `tests/data/test_fvg_labeller.py` fixtures `test_first_candle_always_zero` and `test_last_candle_always_zero`.

**Limitations.**

- Single instrument (SPY). The labeller and pipeline generalise to any OHLCV stream, but model weights and class-weight calibration are SPY-specific.
- The Valid FVG criteria include a 50-bar swing-midpoint check and a BOS lookback. The exact lookback lengths are hyperparameters that were not swept; a different choice might change the positive rate and the difficulty of the classification task.
- The gold set (75 bars) is small. A larger annotation set would give a narrower confidence interval around the kappa estimate, though kappa=1.0 is not improvable.

**Next steps.**

- Notebook 02 covers the gold annotation process in detail, including the annotation tool, sampling strategy, and edge cases observed during hand-labelling.
- Notebook 03 covers model training, evaluation, and the rigor sprint results (XGB macro F1 = 0.721, CNN-LSTM = 0.614, LSTM = 0.599 across five seeds each).

**LO mapping.**

- **LO1 (Data Preparation).** This notebook is the primary LO1 artefact. It covers source selection with rejection rationale (Sections 3, 4), schema and quality validation (Sections 2, 5), session handling (Section 6), labelling and gold annotation (Sections 8, 9), temporal split with regime analysis (Sections 10, 11), windowing strategy with effective-sample reasoning (Section 12), class-weight derivation and persistence (Section 13), and no-lookahead enforcement (Section 14).
- **LO2 (Evaluation choice).** Macro F1 as primary metric is justified in Notebook 00 Section 6 and referenced in Section 8 here via the class-weight discussion.
- **LO3 (Methodology).** CRISP-DM Data Understanding and Data Preparation phases are followed explicitly. Source rejection decisions are evidenced by the validation tests in `tests/data/`.
- **LO4 (Reflection).** The regime-shift observation in Section 10 (2022 bear market as validation set by design, not by accident) reflects a deliberate choice that I made after seeing the class-rate drift in Section 11.

**References.**

- Alpaca Markets. (2026). *Historical market data API, Stocks v2.* https://docs.alpaca.markets/. Retrieved 2026-05-08.
- TradingLab. (2024). *Valid Fair Value Gap detection: a six-criteria rule.* Community documentation. Retrieved 2026-05-08.
- scikit-learn developers. (2024). *Cohen's kappa: sklearn.metrics.cohen\_kappa\_score.* https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html. Retrieved 2026-05-13.
- Brownlee, J. (2017). *Why one-hot encode data in machine learning?* Machine Learning Mastery. Referenced for inverse-frequency class weighting rationale.
- exchange\_calendars contributors. (2026). *exchange\_calendars: trading calendar library.* https://github.com/rsheftel/pandas\_market\_calendars. Retrieved 2026-05-08.